In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel("raw_data_synapse.xlsx")

# Define function to calculate lambda based on user profile
def get_lambda(row):
    base_lambda = 0.5

    # Adjust by age_group
    if row['age_group'] == 'Teen':
        base_lambda += 1.5
    elif row['age_group'] == 'Young Adult':
        base_lambda += 1.0
    elif row['age_group'] == 'Adult':
        base_lambda += 0.7
    elif row['age_group'] == 'Senior':
        base_lambda += 0.3

    # Adjust by tech_savviness
    if row['tech_savviness'] == 'High':
        base_lambda += 1.0
    elif row['tech_savviness'] == 'Medium':
        base_lambda += 0.5
    elif row['tech_savviness'] == 'Low':
        base_lambda += 0.1

    # Adjust by action type
    if row['action'] in ['click_product', 'view_product']:
        base_lambda += 2.0
    elif row['action'] in ['load_home', 'scroll']:
        base_lambda += 0.3

    return base_lambda

# Apply the lambda function to each row
df['lambda'] = df.apply(get_lambda, axis=1)

# Generate items_added_to_cart using Poisson distribution
np.random.seed(42)
df['items_added_to_cart'] = df['lambda'].apply(lambda lam: np.random.poisson(lam))

# Drop lambda column if not needed
df.drop(columns=['lambda'], inplace=True)

# Save updated dataset
df.to_excel("updated_behavior_data_poisson.xlsx", index=False)


print("✅ Dataset updated and saved as 'updated_behavior_data_poisson.xlsx'")


✅ Dataset updated and saved as 'updated_behavior_data_poisson.xlsx'


In [10]:
df = pd.read_excel("updated_behavior_data_poisson.xlsx")

In [11]:
print(df.columns)

Index(['age_group', 'tech_savviness', 'interests', 'device', 'action',
       'items_added_to_cart'],
      dtype='object')


In [17]:
import numpy as np
import pandas as pd

# Make sure to load the full dataset
df = pd.read_excel("updated_behavior_data_poisson.xlsx")

# --- STEP 1: Affluence Score (Poisson) ---
def poisson_lambda(row):
    base = 1.5

    if row['tech_savviness'] == 'High':
        base += 0.7
    if row['age_group'] == 'Senior':
        base -= 0.5
    if row['items_added_to_cart'] > 5:
        base += 1.0
    if 'Luxury' in str(row['interests']):
        base += 1.2
    if row['action'] in ['purchase', 'checkout']:
        base += 0.8

    return max(base, 0.5)  # avoid lambda < 0

df['affluence_score'] = df.apply(lambda row: np.random.poisson(poisson_lambda(row)), axis=1)

def map_affluence(score):
    if score >= 5:
        return "High"
    elif score >= 3:
        return "Medium"
    else:
        return "Low"

df['affluence_level'] = df['affluence_score'].apply(map_affluence)

# --- STEP 2: Consumer Trait (Binomial) ---

def binomial_trait(row):
    prob = 0.3

    # Modify base probability by user traits
    if row['items_added_to_cart'] == 0:
        return "Window Seeker"
    if row['tech_savviness'] == 'High':
        prob += 0.2
    if row['age_group'] == 'Young Adult':
        prob += 0.1
    if 'Finance' in str(row['interests']):
        prob += 0.1
    if 'Luxury' in str(row['interests']):
        prob += 0.15
    if row['device'] == 'Mobile':
        prob -= 0.1
    if row['action'] in ['purchase', 'checkout']:
        prob += 0.3

    # Generate a score
    rand_val = np.random.binomial(n=1, p=min(prob, 0.95))

    # Map result
    if row['items_added_to_cart'] >= 5 and rand_val:
        return "Luxury Purchases"
    elif row['items_added_to_cart'] >= 3:
        return "Brand Conscious"
    elif row['items_added_to_cart'] == 1 and rand_val:
        return "Deal Seeker"
    elif rand_val:
        return "Normal Buyer"
    else:
        return "Window Seeker"

df['consumer_trait'] = df.apply(binomial_trait, axis=1)
df.to_excel("enriched_behavior_data.xlsx", index=False)


In [18]:
print(df.columns)

Index(['age_group', 'tech_savviness', 'interests', 'device', 'action',
       'items_added_to_cart', 'affluence_score', 'affluence_level',
       'consumer_trait'],
      dtype='object')
